# 📓 vLLM 性能基准测试笔记本

本笔记本用于在本地系统上对 vLLM 进行性能基准测试，帮助您量化评估推理性能指标。

# ⚙️ 运行本笔记本的系统要求

为确保测试顺利进行，您的**测试系统**需满足以下配置要求：

## 1. 硬件要求
- **NVIDIA GPU**（H200、GH200、A100 等企业级 GPU）  
- 根据模型规模配置足够的 **GPU 显存**：  
  - 7B 参数模型 → 至少 16–24 GB 显存  
  - 13B 参数模型 → 至少 32–40 GB 显存  
  - 70B 参数模型 → 建议 8× 80 GB GPU（需启用张量并行）  

## 2. 软件环境
- **操作系统**：推荐 Ubuntu 20.04 或 22.04 LTS  
- **NVIDIA 驱动**：版本 535 或更高（使用 `nvidia-smi` 验证）  
- **CUDA 工具包**：版本 12.2 或更高  
- **Python**：3.9 或更新版本  
- **Pip**：保持最新版本（执行 `pip install --upgrade pip`）  

## 3. Python 依赖包（笔记本将自动安装）
- `vllm` - 高性能推理引擎  
- `openai` - 用于与 vLLM 服务交互的 API 客户端  
- `pandas` - 用于结构化存储基准测试结果  
- `matplotlib` - 用于绘制吞吐量曲线图  
- `tqdm` - 进度条显示（可选）  

## 4. 环境配置步骤
- 安装 JupyterLab 或 Jupyter Notebook：
  ```bash
  pip install jupyterlab
  ```
- 启动 Jupyter 服务：
  ```bash
  jupyter lab
  ```
- 在浏览器中打开 `vllm_benchmark_notebook.ipynb` 文件  


In [ ]:

# 步骤 1：安装必需的 Python 依赖包
# 注意：首次安装 vllm 可能需要较长时间，因为需要编译 CUDA 扩展
%pip install vllm openai pandas matplotlib tqdm


In [ ]:

# 步骤 2：启动 vLLM 推理服务器（根据实际情况修改以下配置）
import subprocess, time, os, signal

# 启动 vLLM API 服务器进程
server = subprocess.Popen([
    "python3", "-m", "vllm.entrypoints.api_server",
    "--model", "meta-llama/Llama-2-7b-hf",  # 修改为您的模型路径或 HuggingFace 模型 ID
    "--tensor-parallel-size", "1",          # 修改为您的 GPU 数量（张量并行度）
    "--port", "8000"                        # API 服务端口
], preexec_fn=os.setsid)

# 等待服务器完成初始化（模型加载通常需要几秒到几十秒）
time.sleep(10)
print("✅ vLLM 推理服务已启动，监听端口 :8000")


In [ ]:

# 步骤 3：定义基准测试函数
import time
from openai import OpenAI
import pandas as pd

# 初始化 OpenAI 客户端，指向本地 vLLM 服务
# API key 可以是任意值，因为本地服务通常不需要认证
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

def benchmark(prompt, batch_sizes=[1,4,16,64], requests=20, max_tokens=128):
    """
    执行 vLLM 性能基准测试
    
    参数:
        prompt: 输入提示文本
        batch_sizes: 批次大小列表，用于测试不同并发度下的性能
        requests: 每个批次大小下执行的请求次数
        max_tokens: 每次生成的最大 token 数量
    
    返回:
        包含性能指标的 DataFrame，包括：
        - batch_size: 批次大小
        - avg_ttft: 平均首 token 延迟（Time To First Token）
        - avg_latency: 平均总延迟
        - throughput_tokens_per_s: 吞吐量（tokens/秒）
    """
    results = []
    for b in batch_sizes:
        latencies, tftt = [], []
        for _ in range(requests):
            start = time.time()
            # 发送补全请求，通过重复 prompt 模拟批量请求
            resp = client.completions.create(
                model="model",
                prompt=[prompt]*b,
                max_tokens=max_tokens,
            )
            first_token = time.time()
            tftt.append(first_token - start)
            latencies.append(time.time() - start)

        # 计算当前批次大小的性能指标
        results.append({
            "batch_size": b,
            "avg_ttft": sum(tftt)/len(tftt),
            "avg_latency": sum(latencies)/len(latencies),
            "throughput_tokens_per_s": (requests*b*max_tokens)/sum(latencies)
        })
    return pd.DataFrame(results)


In [ ]:

# 步骤 4：执行基准测试（根据实际需求修改参数）
df = benchmark(
    prompt="解释为什么 GPU 对 AI 工作负载至关重要。",  # 修改为您的测试提示文本
    batch_sizes=[1,4,16,64],                          # 修改批次大小列表，测试不同并发场景
    requests=20,                                      # 修改每个批次的请求数，增加可提高统计准确性
    max_tokens=128                                    # 修改生成的 token 数量
)
# 显示测试结果表格
df


In [ ]:

# 步骤 5：绘制吞吐量曲线图
import matplotlib.pyplot as plt

# 配置支持中文显示（如果出现中文乱码，需安装中文字体）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8,6))
plt.plot(df["batch_size"], df["throughput_tokens_per_s"], marker="o", label="吞吐量")
plt.xscale("log")
plt.xlabel("批次大小（Batch Size）")
plt.ylabel("吞吐量（Tokens/秒）")
plt.title("vLLM 吞吐量扩展性能曲线")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:

# 步骤 6：关闭 vLLM 推理服务器
# 向服务器进程组发送终止信号，确保干净地关闭服务
os.killpg(os.getpgid(server.pid), signal.SIGTERM)
print("🛑 vLLM 推理服务已停止")
